# 🎛️ AVIR Studio — Colab GPU + 7 TB Google Drive Studio
### Продакшн рэпа, трэпа и дрилла на мощных GPU (Nvidia T4 / V100 / A100)

Этот блокнот позволяет использовать бесплатные или платные видеокарты Google Colab вместе с вашим Google Workspace (7 ТБ хранилища) для:
- 🎙️ **Разделения вокала и бита** (Demucs AI Stem Separation);
- 🥁 **Нейросетевой генерации битов и мелодий** (Meta MusicGen / Audiocraft);
- 💎 **Автоматического сохранения всех файлов на Google Диск**.

## Шаг 1: Подключение Google Drive (7 TB Workspace)

In [ ]:
from google.colab import drive
import os

# Монтируем ваш Google Диск
drive.mount('/content/drive')

# Создаем рабочие папки AVIR Studio в вашем 7 ТБ хранилище
STUDIO_DIR = '/content/drive/MyDrive/AVIR_Studio'
os.makedirs(f'{STUDIO_DIR}/beats', exist_ok=True)
os.makedirs(f'{STUDIO_DIR}/vocals', exist_ok=True)
os.makedirs(f'{STUDIO_DIR}/stems', exist_ok=True)
os.makedirs(f'{STUDIO_DIR}/lyrics', exist_ok=True)

print(f'✅ Google Drive успешно подключен! Папка проекта: {STUDIO_DIR}')

## Шаг 2: Проверка GPU (Видеокарты)

In [ ]:
!nvidia-smi

## Шаг 3: Установка Demucs (Разделение любого трека на вокал, бас, барабаны и мелодию)

In [ ]:
!pip install -U demucs soundfile
print('✅ Demucs установлен. Готов разделять треки на стемы!')

### Запуск разделения трека:
Положите MP3/WAV файл в `AVIR_Studio/stems` или укажите путь ниже:

In [ ]:
INPUT_TRACK = '/content/drive/MyDrive/AVIR_Studio/stems/my_track.mp3'
# Разделение трека с помощью нейросети htdemucs на GPU
!demucs --two-stems=vocals "$INPUT_TRACK" -o /content/drive/MyDrive/AVIR_Studio/stems/separated

## Шаг 4: Генерация битов и сэмплов с помощью нейросети Audiocraft (MusicGen)

In [ ]:
!pip install git+https://github.com/facebookresearch/audiocraft.git

from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

model = MusicGen.get_pretrained('facebook/musicgen-medium')
model.set_generation_params(duration=15)  # Длительность в секундах

# Промпты для Drill / Trap / Street Rap
prompts = [
    'Dark UK drill beat, 142 bpm, sliding 808 sub bass, fast rolling triplet hi hats, haunting piano melody, aggressive street bounce',
    'Hard Atlanta trap beat, 138 bpm, booming 808, crisp snare, dark synth bells, ambient reverb'
]

res = model.generate(prompts)

for idx, one_wav in enumerate(res):
    audio_write(f'{STUDIO_DIR}/beats/gen_beat_{idx}', one_wav.cpu(), model.sample_rate, strategy='loudness', loudness_compressor=True)

print('✅ Биты сгенерированы и сохранены в ваш Google Диск!')